# DEG → Functional Enrichment

Pre-ranked GSEA on per-(sex × cell type) DEG CSVs followed by a side-by-side pathway heatmap.

All logic lives in `pygenelab.deg_functional_enrichment`. The DEG dict in **Cell A** drives everything — any number of cell types, either gender, any DEG source as long as the CSV has a gene column + signed log-fold-change + adjusted p-value.

In [25]:
%load_ext autoreload
%autoreload 2

import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

PYGENELAB_PARENT = Path("/ix/djishnu/Akanksha/analysis_code/snRNA_TA_muscle_analysis/TA_muscle_code/py_scripts")
if str(PYGENELAB_PARENT) not in sys.path:
    sys.path.insert(0, str(PYGENELAB_PARENT))

import pygenelab as pgl
import pandas as pd
import matplotlib.pyplot as plt

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Cell A — pick the DEG files

`DEG_FILES` maps a label (used as the heatmap column) to a CSV path. Any sex / any cell type / any number of entries.

Each CSV must have:
- a gene column (the first column is auto-renamed to `GENE_COL`)
- a signed log-fold-change column (`LOG2FC_COL`)
- an adjusted p-value column (`PVAL_COL`)

In [ ]:
# --- Sex × fiber type, unfiltered KO DEGs (_nmt variants, Seurat output) ---
# Load the full unfiltered _nmt CSVs once. The Sankey pipeline (Step 2 onward)
# picks Up / Down genes from these by avg_log2FC sign based on DIRECTION_MODE,
# so flipping the toggle below builds the Up-Sankey or the Down-Sankey.
DIRECTION_MODE = "upregulated"   # "upregulated" -> log2FC > 0, "downregulated" -> log2FC < 0

SEURAT_TABLES = "/ix/djishnu/Akanksha/analysis_code/snRNA_TA_muscle_analysis/Seurat_analysis_outputs/tables"
DEG_FILES = {
    "Fast2x_F": f"{SEURAT_TABLES}/Female_Fast IIX_unfiltered_KO_DEGs_nmt.csv",
    "Fast2b_F": f"{SEURAT_TABLES}/Female_Fast IIB_unfiltered_KO_DEGs_nmt.csv",
    "Fast2x_M": f"{SEURAT_TABLES}/Male_Fast IIX_unfiltered_KO_DEGs_nmt.csv",
    "Fast2b_M": f"{SEURAT_TABLES}/Male_Fast IIB_unfiltered_KO_DEGs_nmt.csv",
}

if DIRECTION_MODE not in {"upregulated", "downregulated"}:
    raise ValueError("DIRECTION_MODE must be 'upregulated' or 'downregulated'")

ANALYSIS_LABEL = f"MF_FastIIX_FastIIB_{DIRECTION_MODE}"
SANKEY_DIRECTION = "Up" if DIRECTION_MODE == "upregulated" else "Down"

# Seurat column conventions (first column is unnamed gene index — auto-renamed to GENE_COL)
GENE_COL = "gene_name"
LOG2FC_COL = "avg_log2FC"
PVAL_COL = "p_val_adj"

# --- Female Fast IIX vs IIB (unfiltered, Seurat output) ---
# DEG_FILES = {
#     "IIX": f"{SEURAT_TABLES}/Female_FastIIX_unfiltered_KO_DEGs.csv",
#     "IIB": f"{SEURAT_TABLES}/Female_FastIIB_unfiltered_KO_DEGs.csv",
# }
# ANALYSIS_LABEL = "Female_FastIIX_FastIIB"

# --- Male Fast IIX vs IIB (unfiltered, Seurat output) ---
# DEG_FILES = {
#     "IIX": f"{SEURAT_TABLES}/Male_Fast IIX_unfiltered_KO_DEGs.csv",
#     "IIB": f"{SEURAT_TABLES}/Male_Fast IIB_unfiltered_KO_DEGs.csv",
# }
# ANALYSIS_LABEL = "Male_FastIIX_vs_FastIIB"

# --- PSC scanpy-style DEGs (column names differ) ---
# DEG_FILES = {
#     "IIX": "/ocean/projects/cis240075p/asachan/datasets/TA_muscle/ERCC1_KO_mice/integrated_samples/analysis/male/degs/DEGs_KO_vs_WT_Fast_IIX.csv",
#     "IIB": "/ocean/projects/cis240075p/asachan/datasets/TA_muscle/ERCC1_KO_mice/integrated_samples/analysis/male/degs/DEGs_KO_vs_WT_Fast_IIB.csv",
# }
# ANALYSIS_LABEL = "Male_FastIIX_vs_FastIIB_scanpy"
# GENE_COL = "gene_name"
# LOG2FC_COL = "logfoldchanges"
# PVAL_COL = "pvals_adj"

print(f"  DIRECTION_MODE     = {DIRECTION_MODE}")
print(f"  SANKEY_DIRECTION   = {SANKEY_DIRECTION}")
print(f"  ANALYSIS_LABEL     = {ANALYSIS_LABEL}")
for label, p in DEG_FILES.items():
    print(f"  {label:>20s}: {p}")


## Cell B — pathway source

Pick the GMT and (optionally) keywords to restrict pathway names. The size filter keeps pathways with 15–500 genes by default.

In [27]:
# --- Mouse MSigDB (full) ---
PATHWAYS_GMT = "/ix/djishnu/Akanksha/datasets/gene_sets/mouse/msigdb.v2024.1.Mm.symbols.gmt"
# PSC: /ocean/projects/cis240075p/asachan/datasets/gene_sets/mouse/msigdb.v2024.1.Mm.symbols.gmt
GENE_ORIGIN = "mice"

# OUTPUT_DIR label. The Sankey runs against the full (size-filtered) pathway
# universe — no keyword restriction — so the LLM has the whole landscape to
# bucket into top-level KEGG-BRITE categories. The Step 9 metabolism heatmap
# does its own narrower filter (see METAB_HEATMAP_TERMS below).
PATHWAY_TYPE = "ALL"

PATHWAY_SIZE_MIN = 15
PATHWAY_SIZE_MAX = 500

# --- Step 9 heatmap: curated metabolism pathways × REACTOME / WP / HALLMARK ---
METAB_HEATMAP_TERMS = [
    "Fatty acid metabolism",
    "Xenobiotic metabolism",
    "Pyruvate metabolism",
    "Phospholipid metabolism",
    "Glycogen metabolism",
    "Glycosaminoglycan metabolism",
    "Purine metabolism",
    "Glucose metabolism",
    "Carbohydrate metabolism",
]
METAB_HEATMAP_DBS = ["REACTOME", "WP", "HALLMARK"]

## Output directory

In [16]:
PATHWAY_TYPE_LABEL = "_".join(PATHWAY_TYPE) if isinstance(PATHWAY_TYPE, (list, tuple)) else str(PATHWAY_TYPE)

OUTPUT_DIR = Path(
    "/ix/djishnu/Akanksha/analysis_code/snRNA_TA_muscle_analysis/TA_muscle_code/py_scripts/3_geneset_scores/Output"
) / ANALYSIS_LABEL / PATHWAY_TYPE_LABEL
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(OUTPUT_DIR)

/ix/djishnu/Akanksha/analysis_code/snRNA_TA_muscle_analysis/TA_muscle_code/py_scripts/3_geneset_scores/Output/Female_FastIIX_FastIIB/ALL


## Step 1 — load DEG CSVs

In [17]:
degs = pgl.load_deg_csvs(DEG_FILES, gene_col=GENE_COL)
for label, df in degs.items():
    print(f"  {label}: {df.shape[0]} genes, columns = {list(df.columns)}")

  IIX: 8294 genes, columns = ['gene_name', 'p_val', 'avg_log2FC', 'pct.1', 'pct.2', 'p_val_adj']
  IIB: 8071 genes, columns = ['gene_name', 'p_val', 'avg_log2FC', 'pct.1', 'pct.2', 'p_val_adj']


## Step 2 — pre-ranked gene lists per DEG file

Ranking = signed log-fold-change × −log10(adjusted p-value).

In [ ]:
RANK_TOP_N = 500  # cap genes per cell type fed to GSEA (top-N by |signed score|)

# pick Up (log2FC > 0) or Down (log2FC < 0) genes from each unfiltered DEG df
# based on DIRECTION_MODE. degs (full) is kept untouched so Step 9 still sees
# both directions.
_sign_keep = (lambda fc: fc > 0) if DIRECTION_MODE == "upregulated" else (lambda fc: fc < 0)
degs_directional = {
    label: df[_sign_keep(df[LOG2FC_COL])].copy()
    for label, df in degs.items()
}
for label, df in degs_directional.items():
    print(f"  {label}: {df.shape[0]} {DIRECTION_MODE} genes (log2FC sign filter)")

ranked_lists = {
    label: pgl.create_ranked_genelist(
        df,
        log2fc_col=LOG2FC_COL,
        pval_col=PVAL_COL,
        gene_col=GENE_COL,
        top_n=RANK_TOP_N,
    )
    for label, df in degs_directional.items()
}
for label, r in ranked_lists.items():
    print(f"  {label}: ranked shape = {r.shape}")


## Step 3 — load + filter pathways

In [19]:
pathways_all = pgl.convert_gmt_to_decoupler_format(PATHWAYS_GMT, gene_origin=GENE_ORIGIN)
pathways = pgl.filter_pathways(
    pathways_all,
    size_min=PATHWAY_SIZE_MIN,
    size_max=PATHWAY_SIZE_MAX,
)
print(f"  pathways retained (size-filtered, no keyword restriction): {pathways['source'].nunique()}")

  pathways retained (size-filtered, no keyword restriction): 9835


## Step 4 — GSEA per DEG file

In [20]:
gsea_results = {label: pgl.run_gsea(r, pathways) for label, r in ranked_lists.items()}
for label, res in gsea_results.items():
    print(f"{label}: {res.shape[0]} pathways scored")
    display(res.head(3))

IIX: 1975 pathways scored


,source,score,pval
0,GOBP_MUSCLE_CELL_DEVELOPMENT,-1.761081,0.0
1,GOBP_SARCOMERE_ORGANIZATION,-1.830988,0.0
2,DESCARTES_ORGANOGENESIS_MYOCYTES,-1.896303,0.0


IIB: 2105 pathways scored


,source,score,pval
0,TORCHIA_TARGETS_OF_EWSR1_FLI1_FUSION_DN,1.684072,0.0
1,WP_AMINO_ACID_METABOLISM,1.665950,0.0
2,MIR_7649_5P,-1.738983,0.0


## Step 5 — annotate + dedup pathway terms

In [21]:
plot_dfs = {
    label: pgl.dedup_by_simplified_term(pgl.prepare_gsea_plot_df(res))
    for label, res in gsea_results.items()
}
for label, df in plot_dfs.items():
    print(f"  {label}: {df.shape[0]} unique simplified terms")

  IIX: 937 unique simplified terms
  IIB: 947 unique simplified terms


## Step 7 — LLM-categorize shared pathways enriched in Fast IIX (Up) → CSV

Buckets every shared, Up-direction pathway in `plot_dfs["IIX"]` into KEGG-BRITE-style top-level categories (Metabolism / Genetic Information Processing / Environmental Information Processing / Cellular Processes / Organismal Systems / Cancer related / Other) using the Claude API. This is a mouse dataset, so disease-named pathways are routed into `Organismal Systems`, except cancer/tumor/oncogene-named ones which land in `Cancer related`. Strips the DB prefix, keeps score/pval, and writes a CSV suitable for a downstream Sankey (DB → category → pathway).

Requires `anthropic` SDK and the `ANTHROPIC_API_KEY` env var. Set `SANKEY_LABEL` / `SANKEY_DIRECTION` to switch cell type or direction.

In [22]:
import os
from pathlib import Path

# Load Anthropic API key from a file outside the repo (one line, key only).
# Create it once: `echo "sk-ant-..." > ~/.anthropic_key && chmod 600 ~/.anthropic_key`
ANTHROPIC_KEY_FILE = Path("~/.anthropic_api_key").expanduser()
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_KEY_FILE.read_text().strip()
print(f"  loaded ANTHROPIC_API_KEY from {ANTHROPIC_KEY_FILE}")

  loaded ANTHROPIC_API_KEY from /ihome/djishnu/aks203/.anthropic_api_key


In [ ]:
SANKEY_LABEL = "Fast2x_F"   # which entry in plot_dfs to pool for this single-label CSV
# SANKEY_DIRECTION is inherited from Cell A (derived from DIRECTION_MODE)
CATEGORIZE_MODEL = "claude-haiku-4-5-20251001"

shared = pgl.common_pathways(plot_dfs)
print(f"  shared pathways across all labels: {len(shared)}")

sankey_df = pgl.categorize_enriched_plot_df(
    plot_dfs[SANKEY_LABEL],
    shared=shared,
    direction=SANKEY_DIRECTION,
    model=CATEGORIZE_MODEL,
)
print(f"  {SANKEY_LABEL} {SANKEY_DIRECTION}-enriched (shared) pathways: {len(sankey_df)}")
print(sankey_df["category"].value_counts())

csv_path = OUTPUT_DIR / f"{ANALYSIS_LABEL}_{PATHWAY_TYPE_LABEL}_{SANKEY_LABEL}_{SANKEY_DIRECTION}_categorized.csv"
sankey_df.to_csv(csv_path, index=False)
print(f"  saved -> {csv_path}")
sankey_df.head(10)


## Step 8 — Sankey: per-list enriched pathways → LLM categories

For each label in `plot_dfs` (e.g. IIX, IIB), pool the shared `SANKEY_DIRECTION`-enriched pathways, classify each via the Claude API, and render a two-column Sankey:

- **Left bar** = source DEG list (one stack per cell type / fiber type).
- **Right bar** = KEGG-BRITE-style top-level category.
- **Flow width** = number of enriched pathways from that list mapped to that category.

Rebuilds the per-label categorized frames inline (independent of Step 7's single-label CSV), then calls `pgl.plot_sankey`.

In [ ]:
SANKEY_LABELS = list(plot_dfs.keys())   # e.g. ["Fast2x_F", "Fast2b_F", "Fast2x_M", "Fast2b_M"]
# SANKEY_DIRECTION is inherited from Cell A (derived from DIRECTION_MODE)
CATEGORIZE_MODEL = "claude-haiku-4-5-20251001"

# match the M/F palette used by the snRNA_related cell-type Sankey
# (F = #e8a0a0, M = #5dbdb2). Any label suffixed `_F` / `_M` inherits the sex color.
SEX_COLORS = {"F": "#e8a0a0", "M": "#5dbdb2"}
left_colors = {
    label: SEX_COLORS.get(label.rsplit("_", 1)[-1])
    for label in SANKEY_LABELS
}

shared = pgl.common_pathways(plot_dfs)

sankey_frames = []
for label in SANKEY_LABELS:
    df_lab = pgl.categorize_enriched_plot_df(
        plot_dfs[label],
        shared=shared,
        direction=SANKEY_DIRECTION,
        model=CATEGORIZE_MODEL,
    )
    df_lab = df_lab.assign(Source=label)
    sankey_frames.append(df_lab)

sankey_long = pd.concat(sankey_frames, ignore_index=True)
print(f"  rows fed to sankey: {len(sankey_long)}")
print(sankey_long.groupby(["Source", "category"]).size().unstack(fill_value=0))

sankey_fig, sankey_ax = pgl.plot_sankey(
    sankey_long,
    left_col="Source",
    right_col="category",
    left_order=SANKEY_LABELS,
    right_order=pgl.PATHWAY_TOP_CATEGORIES,
    left_colors=left_colors,
    title=f"{ANALYSIS_LABEL} — {PATHWAY_TYPE_LABEL} ({SANKEY_DIRECTION}-enriched pathways → categories)",
)
sankey_fig.savefig(
    OUTPUT_DIR / f"{ANALYSIS_LABEL}_{PATHWAY_TYPE_LABEL}_{SANKEY_DIRECTION}_sankey.svg",
    bbox_inches="tight",
)


## Step 9 — Heatmap of curated metabolism pathways (full DEG list, no top-N cap)

Independent of the Sankey pipeline. Filters `pathways_all` down to a curated metabolism set — `METAB_HEATMAP_TERMS` matched across `METAB_HEATMAP_DBS` (REACTOME / WP / HALLMARK) — and re-runs pre-ranked GSEA against **all DEGs** per Fast IIB / Fast IIX (no `RANK_TOP_N` cap). Renders the two-panel Up (red) / Down (blue) `-log10(pval)` heatmap as in the historical `Enriched_Pathways.ipynb` (commit `eaed757`).

In [ ]:
# ---- 1. curate metabolism pathways across REACTOME / WP / HALLMARK ---------
def _term_variants(term):
    """Underscore-uppercase variants of an '<X> metabolism' phrase to match
    both HALLMARK/WP form (X_METABOLISM) and REACTOME form (METABOLISM_OF_X[S])."""
    parts = term.upper().replace("-", "_").split()
    if parts and parts[-1].rstrip("S") == "METABOLISM":
        body = "_".join(parts[:-1])
        return [
            f"{body}_METABOLISM",
            f"METABOLISM_OF_{body}",
            f"METABOLISM_OF_{body}S",
        ]
    return ["_".join(parts)]

_metab_variants = sorted({v for t in METAB_HEATMAP_TERMS for v in _term_variants(t)})

def _is_metab_pathway(name):
    upper = name.upper()
    db = upper.split("_", 1)[0]
    if db not in METAB_HEATMAP_DBS:
        return False
    return any(v in upper for v in _metab_variants)

metab_sources = sorted({s for s in pathways_all["source"].unique() if _is_metab_pathway(s)})
print(f"  curated metabolism pathways retained: {len(metab_sources)}")
for s in metab_sources:
    print(f"    - {s}")

pathways_metab = pathways_all[pathways_all["source"].isin(metab_sources)].copy()

# ---- 2. re-rank DEGs with NO top-N cap (uses all DEGs) ---------------------
ranked_lists_full = {
    label: pgl.create_ranked_genelist(
        df,
        log2fc_col=LOG2FC_COL,
        pval_col=PVAL_COL,
        gene_col=GENE_COL,
        top_n=None,
    )
    for label, df in degs.items()
}
for label, r in ranked_lists_full.items():
    print(f"  {label} (uncapped): ranked shape = {r.shape}")

# ---- 3. re-run GSEA against the curated metabolism set ---------------------
gsea_results_metab = {label: pgl.run_gsea(r, pathways_metab) for label, r in ranked_lists_full.items()}

plot_dfs_metab = {
    label: pgl.dedup_by_simplified_term(pgl.prepare_gsea_plot_df(res))
    for label, res in gsea_results_metab.items()
}
for label, df in plot_dfs_metab.items():
    print(f"  {label}: {df.shape[0]} unique simplified metab terms")

# ---- 4. heatmap: shared pathways, Up = red / Down = blue --------------------
shared_metab = pgl.common_pathways(plot_dfs_metab)
print(f"  shared metab pathways across labels: {len(shared_metab)}")

heatmap_fig, panels = pgl.plot_pathway_heatmap(
    plot_dfs_metab,
    pathways=shared_metab,
    title=f"{ANALYSIS_LABEL} — curated metabolism pathways",
)
heatmap_fig.savefig(OUTPUT_DIR / f"{ANALYSIS_LABEL}_metab_heatmap.png", dpi=300, bbox_inches="tight")
heatmap_fig.savefig(OUTPUT_DIR / f"{ANALYSIS_LABEL}_metab_heatmap.svg", bbox_inches="tight")
heatmap_fig